In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common import *

In [ ]:
set_seed(1)
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = ["blue", "red", "green"] * 10

In [ ]:
c1 = 0.5
c2 = 1
f1 = 2 * torch.pi / 24
f2 = 8 * 2 * torch.pi / 24
def sample(batch, sizes, device, mul=0.01):
    k = torch.arange(sizes, device=device).unsqueeze(dim=0).repeat(batch, 1).float()
    b1 = (torch.rand((batch, 1), device=device).round() * 2 - 1)
    b2 = (torch.rand((batch, 1), device=device).round() * 2 - 1)
    x = b1 * c1 * torch.sin(k * f1) + b2 * c2 * torch.sin(k * f2)
    epsilon1 = mul * torch.randn_like(x)
    epsilon2 = mul * torch.randn_like(x)
    return x + epsilon1, x + epsilon2

def sample_with_rand(batch, sizes, device, mul=0.01):
    k = torch.arange(sizes, device=device).unsqueeze(dim=0).repeat(batch, 1).float()
    b1 = (torch.rand((batch, 1), device=device).round() * 2 - 1)
    b2 = (torch.rand((batch, 1), device=device).round() * 2 - 1)
    x = b1 * c1 * torch.sin(k * f1) + b2 * c2 * torch.sin(k * f2)
    epsilon1 = mul * torch.randn_like(x)
    epsilon2 = mul * torch.randn_like(x)
    return x + epsilon1, x + epsilon2, torch.cat([(b1+1)/2, (b2+1)/2], dim=-1)

In [ ]:
def get_e(sizes, device):
    k = torch.arange(sizes, device=device).unsqueeze(dim=0).float()
    e1 = c1 * torch.sin(k * f1)
    e2 = c2 * torch.sin(k * f2)
    return torch.stack([e1, e2], dim=0)

In [ ]:
sizes = 96
samples = sample(1, sizes, "cuda")
# plt.xlim(0, sizes-1)
fig, axs = plt.subplots(1, 1, figsize=(14, 4))
axs.plot(samples[0][0].cpu(), color=colors[0])
axs.set_xlabel(r"$t$")
axs.set_ylabel(r"$x$")
# plt.savefig("figure/input.pdf", format="pdf", bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
batch = 2000
epochs = 2000

s_in = sizes
s_mid = s_in // 2
s_out = 2
device = torch.device("cuda")
inner_layer = []
for i in range(2):
    inner_layer.append(nn.Linear(s_mid, s_mid))
    inner_layer.append(nn.LeakyReLU())
simple_network = nn.Sequential(nn.Linear(s_in, s_mid), nn.LeakyReLU(), *inner_layer, nn.Linear(s_mid, s_out)).to(device)
lr_simple = 1e-4
lossFn = BTLoss(in_feature=s_out, use_batchnorm=True, lambd=0.5).to("cuda")


def evaluate(model, data):
    x0, x1 = data
    z0, z1 = torch.chunk(model(torch.cat([x0.clone(), x1.clone()], dim=0)), 2, dim=0)
    c, _ = lossFn(z0, z1)
    eigvals = torch.linalg.eigvalsh(c)
    return eigvals.max()


# weights_init_scale, top_c_eigval = find_sufficiently_small_init(simple_network, eval_fn=evaluate, data=sample(batch, sizes, device),
#                                                                 eval_target=1e-2)
# scale_weights(simple_network, weights_init_scale)
scale = 2e-1
scale_weights(simple_network, scale)
t = sample(batch, sizes, device)
gamma = t[0].T @ t[1]  # n x n
gamma = (gamma + gamma.T) / (2 * batch)
plt.imshow(gamma.cpu())
plt.tight_layout()
plt.colorbar()
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)

In [ ]:
print(gamma_l[0:4])
fig, axs = plt.subplots(1, 3, figsize=(24, 4))
for i in range(3):
    axs[i].plot(gamma_v.T[i].cpu(), color=colors[i])
    # axs[i].set_xlim(0, sizes-1)
    axs[i].set_xlabel(r"$v_" + str(i + 1) + r"\; \text{index}$")
    # axs[i].set_ylabel("Value")
plt.tight_layout()
# plt.savefig(f"figure/gamma_eigenvectors.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
optimizer = torch.optim.AdamW(simple_network.parameters(), lr_simple, weight_decay=0)

loss_values = []
eiglist = [[] for i in range(s_out)]
ones = torch.ones(s_in, device=device)
zeros = torch.zeros(s_in, device=device)
e = get_e(sizes, device)

print("e", e)
weight_center = None
eig_reversed = False
tt = []
looper = tqdm(range(1, epochs + 1))

linear = nn.Sequential(nn.Linear(s_out, s_mid), nn.ReLU(), nn.Linear(s_mid, s_out), nn.Sigmoid()).to(device)
linear_optimizer = torch.optim.AdamW(linear.parameters(), 1e-3)

corrects = [[] for i in range(s_out)]
loss_decrease = False
for epoch in looper:
    optimizer.zero_grad()
    simple_network.train()
    x0, x1 = sample(batch, sizes, device)
    z0, z1 = torch.chunk(simple_network(torch.cat([x0.clone(), x1.clone()], dim=0)), 2, dim=0)
    c, loss = lossFn(z0, z1)
    c = c.detach()
    if epoch == 1:
        print("output", c)
    v = reversed(torch.linalg.eigvalsh(c))
    for i in range(len(v)):
        eiglist[i].append(v[i].cpu().item())
    loss_values.append(loss.item())
    loss.backward()
    optimizer.step()
    looper.set_postfix({"loss": loss.item()})


    simple_network.eval()
    linear.train()
    for i in range(100):
        linear_optimizer.zero_grad()
        with torch.no_grad():
            x0, x1, rands = sample_with_rand(batch, sizes, device)
            z = simple_network(torch.cat([x0, x1], dim=0))
        z = linear(z)
        label = torch.cat([rands, rands], dim=0)
        loss = F.mse_loss(z, label)
        loss.backward()
        linear_optimizer.step()
        if loss.item() < 0.9 and not loss_decrease:
            loss_decrease = True
            print("loss decrease")

    linear.eval()
    total = 0.0
    correct = [0.0 for i in range(s_out)]
    for i in range(100):
        with torch.no_grad():
            x0, x1, rands = sample_with_rand(batch, sizes, device)
            z = simple_network(torch.cat([x0, x1], dim=0))
            z = linear(z)
        label = torch.cat([rands, rands], dim=0)
        total += label.size(0)
        for j in range(s_out):
            #mse
            correct[j] += torch.sum(((z[:, j] - label[:, j])**2).float()).cpu().item()
    for i in range(s_out):
        corrects[i].append(correct[i] / total)

In [ ]:
palette = plt.rcParams['axes.prop_cycle'].by_key()['color']
alpha = 0.8
fig, axs = plt.subplots(1, 2, figsize=(24, 3.8))

axs[0].plot(loss_values, color=colors[2], linewidth=4, alpha=0.65)
index = []
axs[0].set_ylim(-0.05, s_out * 1.15)
for i in range(len(loss_values)):
    if loss_values[i] - (s_out - len(index) - 0.5) < 0.01:
        index.append(i)
[axs[0].axvline(_x, color='black', linestyle='--') for _x in index]
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Loss")

t = np.array(eiglist)
for i in range(len(eiglist)):
    axs[1].plot(eiglist[i], linewidth=4, alpha=0.65, label=(r"$\lambda_" + ("h$" if i == 0 else "l$")), color=colors[i])
[axs[1].axvline(_x, color='black', linestyle='--') for _x in index]
axs[1].set_xlabel("Step")
axs[1].set_ylabel("Eigenvalue")
axs[1].legend()

plt.savefig("figure/freq_loss_eig.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
palette = plt.rcParams['axes.prop_cycle'].by_key()['color']
alpha = 0.8
fig, axs = plt.subplots(1, 3, figsize=(24, 3.8))

axs[0].plot(loss_values, color=colors[2], linewidth=4, alpha=0.65)
index = []
axs[0].set_ylim(-0.05, s_out * 1.15)
for i in range(len(loss_values)):
    if loss_values[i] - (s_out - len(index) - 0.5) < 0.01:
        index.append(i)
[axs[0].axvline(_x, color='black', linestyle='--') for _x in index]
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Loss")

t = np.array(eiglist)
for i in range(len(eiglist)):
    axs[1].plot(eiglist[i], linewidth=4, alpha=0.65, label=(r"$\lambda_" + ("h$" if i == 0 else "l$")), color=colors[i])
[axs[1].axvline(_x, color='black', linestyle='--') for _x in index]
axs[1].set_xlabel("Step")
axs[1].set_ylabel("Eigenvalue")
axs[1].legend()
for i in reversed(range(s_out)):
    q = np.array(corrects[i])
    axs[2].plot(q, linewidth=4, alpha=0.65, label=(r"${\text{FLD}(e_l)}$" if i == 0 else r"${\text{FLD}(e_h)}$"), color=colors[1-i])
axs[2].legend(loc='lower left')
[axs[2].axvline(_x, color='black', linestyle='--') for _x in index]
plt.savefig("figure/freq_loss_eig_fld.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
def smooth_data(data, window_size=5):
    data_array = np.array(data)
    smooth_data = np.copy(data_array)
    half_window = window_size // 2
    for i in range(len(data_array)):
        start_idx = max(0, i - half_window)
        end_idx = min(len(data_array), i + half_window + 1)
        smooth_data[i] = np.mean(data_array[start_idx:end_idx])
    return smooth_data

In [ ]:
##dc/dt, -dl/dt
fig, axs = plt.subplots(1, 1, figsize=(14, 3.7))
for i in range(2):
    plt.plot(smooth_data(dt(eiglist[i])) / lr_simple, alpha=0.5, linewidth=6, color=colors[i],
             label=r"$d\lambda_" + ("h" if i == 0 else "l") + "/dt$")

dlossdt = -1 * np.array(dt(loss_values))
axs.plot(smooth_data(dlossdt / lr_simple), alpha=1, linestyle='--', color='black', linewidth=1.5, label=r"$-d\mathcal{L}/dt$")

axs.set_yticks([-60, 0, 60, 120, ])

# axs.set_ylim(-0.025, epochs)
axs.set_xlim(0, epochs)
axs.set_xlabel("Step")
# axs.set_ylabel("Value")
axs.legend(loc='upper right')
plt.savefig("figure/freq_dcdt_dldt.pdf", format="pdf", bbox_inches='tight')
plt.show()